# kvroute · Phase 3: the router on real vLLM workers

Starts 2 vLLM workers (one per T4), checks each one, then replays the same multi-turn image trace through the router once per policy (round-robin, least-requests, least-loaded, session-affinity, cache-aware) at two load levels, resetting the workers' caches between policies. Ends with a validation report.

**Time:** about 75-90 minutes (5 policies x 2 loads x ~6.5 min, plus setup). To halve it, use `--convs 24` (one load level) or drop a policy with `--policies round_robin,least_requests,session_affinity,cache_aware`.

**Settings:** *Accelerator* = **GPU T4 x2**, **Internet on**, optionally attach the COCO 2017 dataset. Run all cells; download `phase3.zip` at the end.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,compute_cap --format=csv
!python --version

In [ ]:
!pip install -q "vllm==0.32.0" 2>&1 | tail -3
!python -c "import vllm, torch; print('vllm', vllm.__version__, 'torch', torch.__version__, 'cuda', torch.version.cuda)"

In [ ]:
BRANCH = 'phase-3-router'
%cd /kaggle/working
!rm -rf kvsim && git clone -q -b {BRANCH} https://github.com/mansidhamne/kvsim.git
%cd /kaggle/working/kvsim
!git log --oneline -1
!pip install -q -e ".[router]" 2>&1 | tail -2

In [ ]:
import glob
found = sorted(glob.glob('/kaggle/input/**/val2017', recursive=True))
IMAGES = found[0] if found else ''
print('images:', IMAGES or 'synthetic')

In [ ]:
# The whole experiment. Prints one line per (load, policy) and a validation report at the end.
img_flag = f'--images {IMAGES}' if IMAGES else ''
!python -m deploy.kaggle_run phase3 --out /kaggle/working/phase3 --workers 2 --gpus 2 --convs 12,24 {img_flag}

In [ ]:
from IPython.display import Image, display
display(Image('/kaggle/working/phase3/phase3.png'))

In [ ]:
%cd /kaggle/working
!zip -qr phase3.zip phase3 && ls -lh phase3.zip

Download `phase3.zip` and unzip it into `results/phase3/` in the repo. If a worker fails its smoke check, the reason is in `phase3/smoke_failed.json` and `phase3/logs/`.